#### Dense Vector field type stores dense vectors of numeric values, primarily used for `k-nearest neighbor (kNN)` search. It doesn't support aggregation or sorting, `dense vector` field can be added as an array of numeric values based on element_type with float by default.

### Connect to ElasticSearch

In [2]:
from pprint import pprint
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
client_info = es.info()
pprint('Connected to Elasticsearch successfully!')
pprint(client_info.body)

'Connected to Elasticsearch successfully!'
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'WGXTdf8bTw6Y1ejhBBncsA',
 'name': 'c813a54bbd9a',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2024-08-05T10:05:34.233336849Z',
             'build_flavor': 'default',
             'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '9.11.1',
             'minimum_index_compatibility_version': '7.0.0',
             'minimum_wire_compatibility_version': '7.17.0',
             'number': '8.15.0'}}


### Inserting Documents
- manual mapping require with dense_vector data types

In [3]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(
    index="my_index",
    mappings={
        "properties": {
            "sides_length": {
                "type": "dense_vector",
                "dims": 6
            },
            "shape": {
                "type": "keyword"
            }
        }
    },
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

In [4]:
from pprint import pprint

response = es.index(
    index='my_index',
    id=1,
    document={
        "shape": "square",
        "sides_length": [2, 3, 4, 5, 7, 8],
    }
)

pprint(response.body)

{'_id': '1',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 0,
 '_shards': {'failed': 0, 'successful': 1, 'total': 2},
 '_version': 1,
 'result': 'created'}


sides_length is a dense_vector with 4 dimensions and cosine set as the similarity measure. These are parameters we can modify if needed. Since we didn’t specify them when manually setting the mapping, Elasticsearch automatically applied the default values.

In [5]:
pprint(es.indices.get_mapping(index='my_index').body)

{'my_index': {'mappings': {'properties': {'shape': {'type': 'keyword'},
                                          'sides_length': {'dims': 6,
                                                           'index': True,
                                                           'index_options': {'ef_construction': 100,
                                                                             'm': 16,
                                                                             'type': 'int8_hnsw'},
                                                           'similarity': 'cosine',
                                                           'type': 'dense_vector'}}}}}


### Indexing a matrix not supported with the `dense vector` field type 
- The below code throws error

In [9]:
response = es.index(
    index='my_index',
    id=2,
    document={
        "shape": "square",
        "sides_length": [[5, 5], [5, 5]],
    }
)
pprint(response.body)

BadRequestError: BadRequestError(400, 'document_parsing_exception', 'Failed to parse object: expecting token of type [VALUE_NUMBER] but found [START_ARRAY]', Failed to parse object: expecting token of type [VALUE_NUMBER] but found [START_ARRAY])